<!-- # Simulations - In Sample -->

In [ ]:
# ## =====================================================================
# ## 0) Setup
# ## =====================================================================
# req_pkgs <- c("devtools","ggplot2","dplyr","tidyr","tibble","scales")
# need <- setdiff(req_pkgs, rownames(installed.packages()))
# if (length(need)) install.packages(need, dependencies = TRUE)
# invisible(lapply(req_pkgs, require, character.only = TRUE))

# # Load your local package so qdesn_fit_vb(), forecast_paths.qdesn_fit(), etc. are available
# devtools::load_all("/home/antonio/code/exdqlm")

# ## =====================================================================
# ## 1) Paths & Data Files
# ## =====================================================================
# base_dir  <- "/home/antonio/code/exdqlm/results/sim_suite_dlm/series/dlm_constV_smallW"
# file_long <- file.path(base_dir, "series_long.csv")  # columns: t, p, q, y, mu
# stopifnot(file.exists(file_long))

# ## =====================================================================
# ## 2) User-Tunable Configuration (MODEL • SAMPLING • PLOT • SYNTHESIS)
# ## =====================================================================

# # ---- TARGETED QUANTILES (fit one model per p0) -----------------------
# p_vec <- c(0.025, 0.50, 0.975)

# # ---- DESN/READOUT MODEL SPEC (fixed spec; same reservoir across fits) ----
# desn_args <- list(
#   D = 1L,
#   n = c(500L),
#   n_tilde = integer(0),
#   m = 21L,
#   alpha = 0.6,
#   rho = c(0.90),
#   act_f = "tanh",
#   act_k = "identity",
#   pi_w  = 0.05,
#   pi_in = 1.00,
#   washout = 400L,
#   add_bias = TRUE,
#   seed = 42       # ensures identical reservoir across quantiles
# )

# # Variational Bayes options for exAL static readout
# vb_args <- list(max_iter = 2000, tol = 1e-4, n_samp_xi = 2000, verbose = TRUE)

# # ---- POSTERIOR SAMPLING ---------------------------------------------
# nd_draws <- 3000L
# chunk_sz <- 250L

# # ---- PLOTTING --------------------------------------------------------
# last_window <- 200L
# plot_base   <- 13
# col_map <- setNames(c("#ef4444", "#10b981", "#0ea5e9"), as.character(p_vec))

# # ---- SYNTHESIS (optional, across quantile models) --------------------
# synth_isotonic  <- TRUE
# synth_rearrange <- TRUE
# synth_grid_M    <- 2001L
# synth_nsamp     <- 4000L
# synth_seed      <- 123

# # ---- OPTIONAL SAVE ---------------------------------------------------
# save_outputs <- FALSE
# out_dir <- file.path(base_dir, "qdesn_forecast_outputs")
# if (save_outputs && !dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

# ## =====================================================================
# ## 3) Helpers
# ## =====================================================================
# nearest_p <- function(avail_p, target) avail_p[ which.min(abs(avail_p - target)) ]

# band_from_draws <- function(mat, level = 0.95) {
#   probs <- c((1 - level)/2, 0.5, (1 + level)/2)
#   qs <- t(apply(mat, 1, stats::quantile, probs = probs, names = FALSE))
#   colnames(qs) <- c("lo","med","hi")
#   qs
# }
# fmt_p <- function(x) sprintf("%.3f", as.numeric(x))

# ## =====================================================================
# ## 4) Load data + define Train / Forecast split
# ## =====================================================================
# dat_long <- read.csv(file_long) |>
#   tibble::as_tibble() |>
#   dplyr::mutate(
#     t = as.integer(t),
#     p = as.numeric(p),
#     q = as.numeric(q),
#     y = as.numeric(y),
#     mu = as.numeric(mu)
#   ) |>
#   dplyr::arrange(t, p)

# # Observed y per time (unique)
# y_full <- dat_long |>
#   dplyr::distinct(t, y) |>
#   dplyr::arrange(t)

# T_full <- nrow(y_full)
# p_all  <- sort(unique(dat_long$p))

# # ---- Choose ONE way to split (set the other to NA) -------------------
# H_forecast <- 200L          # e.g., forecast the last 200 time points
# train_prop <- NA_real_      # OR set (e.g., 0.8) and set H_forecast <- NA_real_

# if (!is.na(H_forecast) && !is.na(train_prop)) {
#   stop("Set either H_forecast or train_prop, not both.")
# }
# if (is.na(H_forecast) && is.na(train_prop)) {
#   stop("You must set either H_forecast or train_prop.")
# }

# if (!is.na(train_prop)) {
#   n_train    <- max(1L, floor(train_prop * T_full))
#   H_forecast <- T_full - n_train
# } else {
#   n_train <- T_full - as.integer(H_forecast)
# }

# stopifnot(n_train > 0, H_forecast > 0, n_train + H_forecast == T_full)

# idx_tr <- 1:n_train
# idx_fc <- (n_train + 1):T_full

# y_train    <- y_full$y[idx_tr]
# y_forecast <- y_full$y[idx_fc]

# ## =====================================================================
# ## 5) Fit (Train) and Forecast (Holdout) per-quantile
# ## =====================================================================
# fit_and_forecast_p <- function(p0) {
#   # (a) Fit on TRAIN only with fixed reservoir spec
#   fit_tr <- do.call(
#     qdesn_fit_vb,
#     c(list(y = y_train, p0 = p0, vb_args = vb_args), desn_args)
#   )

#   # (b) Recursive multi-step forecast over the FORECAST horizon
#   m_lag <- as.integer(fit_tr$meta$m)
#   y_hist <- if (m_lag > 0L) tail(y_train, m_lag) else numeric(0)

#   fc <- forecast_paths.qdesn_fit(
#     object = fit_tr,
#     H      = H_forecast,
#     nd     = nd_draws,
#     method = "recursive",
#     y_hist = y_hist,
#     chunk  = chunk_sz
#   )
#   yrep_fc     <- fc$yrep          # H × nd
#   mu_draws_fc <- fc$mu_draws      # H × nd

#   # (c) Summaries for plotting
#   q_pred_fc <- apply(yrep_fc, 1, stats::quantile, probs = p0, names = FALSE)
#   mu_qs_fc  <- band_from_draws(mu_draws_fc, level = 0.95)

#   # TRUE q_p for the forecast window (nearest p in file)
#   p_use <- nearest_p(p_all, p0)
#   q_true_vec <- dat_long |>
#     dplyr::filter(abs(p - p_use) < 1e-12) |>
#     dplyr::arrange(t) |>
#     dplyr::pull(q)
#   stopifnot(length(q_true_vec) == T_full)
#   q_true_fc <- q_true_vec[idx_fc]

#   # tidy frames for forecast-only plots (h = 1..H)
#   df_mu_fc <- tibble::tibble(
#     h         = seq_len(H_forecast),
#     p0        = p0,
#     mu        = mu_qs_fc[, "med"],
#     lo        = mu_qs_fc[, "lo"],
#     hi        = mu_qs_fc[, "hi"],
#     q_true    = q_true_fc,
#     y         = y_forecast
#   )

#   df_pred_fc <- tibble::tibble(
#     h         = seq_len(H_forecast),
#     p0        = p0,
#     q_pred    = q_pred_fc,
#     q_true    = q_true_fc,
#     y         = y_forecast
#   )

#   list(
#     fit_train   = fit_tr,
#     yrep_fc     = yrep_fc,
#     mu_draws_fc = mu_draws_fc,
#     df_mu_fc    = df_mu_fc,
#     df_pred_fc  = df_pred_fc,
#     p_use       = p_use
#   )
# }

# ## =====================================================================
# ## 6) Plotting (forecast window only)
# ## =====================================================================
# plot_mu_band_forecast <- function(df, p0, p_use, window = last_window) {
#   i2 <- max(df$h); i1 <- max(1L, i2 - window + 1L)
#   d  <- dplyr::filter(df, dplyr::between(h, i1, i2))

#   ggplot2::ggplot(d, ggplot2::aes(x = h)) +
#     ggplot2::theme_minimal(base_size = plot_base) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "forecast step (h)", y = "value",
#       title = sprintf("Forecast μ̂ with 95%% band vs TRUE q_p & y  |  p = %s",
#                       scales::percent(p0, accuracy = 1)),
#       subtitle = sprintf("TRUE quantile taken at p = %s (nearest in file).",
#                          scales::percent(p_use, accuracy = 1))
#     ) +
#     ggplot2::geom_ribbon(ggplot2::aes(ymin = lo, ymax = hi),
#                          fill = scales::alpha(col_map[as.character(p0)], 0.25),
#                          colour = NA) +
#     ggplot2::geom_line(ggplot2::aes(y = mu, colour = "mu"), linewidth = 0.9) +
#     ggplot2::geom_line(ggplot2::aes(y = q_true, colour = "true_q"),
#                        linewidth = 0.9, linetype = 2) +
#     ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"),
#                        linewidth = 0.6) +
#     ggplot2::scale_color_manual(
#       name = "",
#       values = c(mu = col_map[as.character(p0)],
#                  true_q = "#7c3aed",
#                  data = "#111111"),
#       breaks = c("mu","true_q","data"),
#       labels = c("μ̂ (forecast)", "true q_p", "observed y")
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# plot_pred_quantile_forecast <- function(df, p0, p_use, window = last_window) {
#   i2 <- max(df$h); i1 <- max(1L, i2 - window + 1L)
#   d  <- dplyr::filter(df, dplyr::between(h, i1, i2))

#   ggplot2::ggplot(d, ggplot2::aes(x = h)) +
#     ggplot2::theme_minimal(base_size = plot_base) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "forecast step (h)", y = "value",
#       title = sprintf("Predictive q_p vs TRUE q_p (forecast)  |  p = %s",
#                       scales::percent(p0, accuracy = 1)),
#       subtitle = sprintf("TRUE quantile taken at p = %s (nearest in file).",
#                          scales::percent(p_use, accuracy = 1))
#     ) +
#     ggplot2::geom_line(ggplot2::aes(y = q_pred, colour = "pred"),
#                        linewidth = 0.9) +
#     ggplot2::geom_line(ggplot2::aes(y = q_true, colour = "true"),
#                        linewidth = 0.9, linetype = 2) +
#     ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"),
#                        linewidth = 0.6, alpha = 0.85) +
#     ggplot2::scale_color_manual(
#       name = "",
#       values = c(pred = col_map[as.character(p0)],
#                  true = "#7c3aed",
#                  data = "#111111"),
#       breaks = c("pred","true","data"),
#       labels = c("empirical q_p (from forecast draws)", "true q_p", "observed y")
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# ## =====================================================================
# ## 7) Run fits for p ∈ p_vec and produce per-p forecast plots
# ## =====================================================================
# fits_fc <- lapply(p_vec, fit_and_forecast_p)
# names(fits_fc) <- paste0("p=", p_vec)

# for (k in seq_along(p_vec)) {
#   p0    <- p_vec[k]
#   p_use <- fits_fc[[k]]$p_use

#   g1 <- plot_mu_band_forecast(fits_fc[[k]]$df_mu_fc,   p0, p_use, window = last_window)
#   g2 <- plot_pred_quantile_forecast(fits_fc[[k]]$df_pred_fc, p0, p_use, window = last_window)
#   print(g1); print(g2)

#   if (isTRUE(save_outputs)) {
#     if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
#     ggsave(file.path(out_dir, sprintf("forecast_mu_band_p=%s.png", as.character(p0))),
#            g1, width = 9, height = 4.8, dpi = 150)
#     ggsave(file.path(out_dir, sprintf("forecast_pred_vs_true_p=%s.png", as.character(p0))),
#            g2, width = 9, height = 4.8, dpi = 150)
#   }
# }

# ## =====================================================================
# ## 8) (Optional) Synthesize across per-p models on the forecast window
# ## =====================================================================
# # Collect forecast predictive draws (H × nd) for each fitted quantile model
# draws_list_fc <- lapply(fits_fc, function(obj) obj$yrep_fc)

# synth_fc <- exdqlm_synthesize_from_draws(
#   draws_list      = draws_list_fc,
#   p               = p_vec,
#   enforce_isotonic = synth_isotonic,
#   rearrange        = synth_rearrange,
#   grid_M           = synth_grid_M,
#   n_samp           = synth_nsamp,
#   seed             = synth_seed,
#   T_expected       = H_forecast
# )
# # synth_fc$draws : (H_forecast × synth_nsamp)

# # Build TRUE q_p for a comparison set on the forecast window
# p_comp     <- c(0.05, 0.50, 0.95)
# p_use_comp <- vapply(p_comp, function(tau) nearest_p(p_all, tau), numeric(1))

# true_fc_mat <- dat_long |>
#   dplyr::filter(t %in% idx_fc, p %in% p_use_comp) |>
#   dplyr::mutate(h = match(t, idx_fc)) |>
#   dplyr::arrange(h, p)

# # Pivot TRUE into wide columns: true_q_0.05, true_q_0.50, true_q_0.95
# true_fc_wide <- true_fc_mat |>
#   dplyr::mutate(lab = paste0("true_q_", fmt_p(vapply(p, identity, numeric(1))))) |>
#   dplyr::select(h, lab, q) |>
#   tidyr::pivot_wider(names_from = lab, values_from = q) |>
#   dplyr::arrange(h)

# # Synthesized quantiles over horizon
# synth_cols_fc <- lapply(p_comp, function(tau)
#   apply(synth_fc$draws, 1L, stats::quantile, probs = tau, names = FALSE))
# names(synth_cols_fc) <- paste0("synth_q_", fmt_p(p_comp))
# synth_q_fc <- tibble::as_tibble(synth_cols_fc)

# # Assemble comparison frame for forecast horizon
# compare_fc <- tibble::tibble(
#   h = seq_len(H_forecast),
#   y = y_forecast
# ) |>
#   dplyr::bind_cols(true_fc_wide |> dplyr::select(-h)) |>
#   dplyr::bind_cols(synth_q_fc)

# # Plot helper for synthesized vs true on forecast window
# plot_synth_vs_true_forecast <- function(df_s, tau, window = last_window) {
#   tau_lab <- fmt_p(tau)
#   c_true  <- paste0("true_q_",  tau_lab)
#   c_synth <- paste0("synth_q_", tau_lab)

#   i2 <- max(df_s$h); i1 <- max(1L, i2 - window + 1L)
#   d  <- dplyr::filter(df_s, dplyr::between(h, i1, i2))

#   ggplot2::ggplot(d, ggplot2::aes(x = h)) +
#     ggplot2::theme_minimal(base_size = plot_base) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "forecast step (h)", y = "value",
#       title = sprintf("Synthesized predictive q_p vs TRUE q_p  |  p = %s",
#                       scales::percent(as.numeric(tau), accuracy = 1))
#     ) +
#     ggplot2::geom_line(ggplot2::aes(y = .data[[c_synth]], colour = "synth"), linewidth = 0.9) +
#     ggplot2::geom_line(ggplot2::aes(y = .data[[c_true]],  colour = "true"),
#                        linewidth = 0.9, linetype = 2) +
#     ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"), linewidth = 0.6, alpha = 0.85) +
#     ggplot2::scale_color_manual(
#       name = "", values = c(synth = "#0ea5e9", true = "#7c3aed", data = "#111111"),
#       breaks = c("synth","true","data"),
#       labels = c("synth q_p", "true q_p", "observed y")
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# # Draw synthesized-vs-true plots per p in p_comp
# plots_synth_fc <- lapply(p_comp, function(tau) {
#   plot_synth_vs_true_forecast(compare_fc, tau, window = last_window)
# })
# for (g in plots_synth_fc) print(g)

# ## =====================================================================
# ## 9) Observed y with synthesized 95% predictive band (forecast window)
# ## =====================================================================
# plot_obs_with_95_band_forecast <- function(synth_draws, y_vec, window = 50L,
#                                            fill_col = "#3B82F6",
#                                            show_median = TRUE) {
#   stopifnot(is.matrix(synth_draws), length(y_vec) == nrow(synth_draws))

#   T_h <- nrow(synth_draws)
#   i2 <- T_h
#   i1 <- max(1L, i2 - as.integer(window) + 1L)

#   q_mat <- t(apply(synth_draws, 1L, stats::quantile,
#                    probs = c(0.025, 0.50, 0.975), names = FALSE))
#   colnames(q_mat) <- c("q025","q50","q975")

#   df <- tibble::tibble(
#     h   = seq_len(T_h),
#     y   = y_vec,
#     q05 = q_mat[, "q025"],
#     q50 = q_mat[, "q50"],
#     q95 = q_mat[, "q975"]
#   ) |>
#     dplyr::filter(dplyr::between(h, i1, i2))

#   ggplot2::ggplot(df, ggplot2::aes(x = h)) +
#     ggplot2::theme_minimal(base_size = 13) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "forecast step (h)", y = "value",
#       title = "Observed y with synthesized 95% predictive band (forecast)",
#       subtitle = sprintf("Band = [2.5%%, 97.5%%]; window: last %d steps", i2 - i1 + 1)
#     ) +
#     ggplot2::geom_ribbon(
#       ggplot2::aes(ymin = q05, ymax = q95),
#       fill   = scales::alpha(fill_col, 0.22),
#       colour = NA
#     ) +
#     { if (isTRUE(show_median))
#         ggplot2::geom_line(ggplot2::aes(y = q50, colour = "median"), linewidth = 0.7)
#       else ggplot2::geom_blank()
#     } +
#     ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"), linewidth = 0.8) +
#     ggplot2::scale_color_manual(
#       name = "", breaks = c("data","median"),
#       values = c(data = "#111111", median = fill_col),
#       labels = c("observed y","predictive median")
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# g_band_fc <- plot_obs_with_95_band_forecast(
#   synth_draws = synth_fc$draws,   # (H_forecast × n_samp)
#   y_vec       = y_forecast,
#   window      = last_window,
#   fill_col    = "#3B82F6",
#   show_median = TRUE
# )
# print(g_band_fc)

# if (isTRUE(save_outputs)) {
#   if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
#   saveRDS(
#     list(fits_fc = fits_fc, synth_fc = synth_fc, compare_fc = compare_fc),
#     file.path(out_dir, "forecast_objects.rds")
#   )
#   for (j in seq_along(plots_synth_fc)) {
#     ggsave(file.path(out_dir, sprintf("forecast_synth_vs_true_p=%s.png", fmt_p(p_comp[j]))),
#            plots_synth_fc[[j]], width = 9, height = 4.8, dpi = 150)
#   }
#   ggsave(file.path(out_dir, "forecast_obs_with_95_band.png"),
#          g_band_fc, width = 9, height = 4.8, dpi = 150)
# }


<!-- # Simulation - Forecast -->

In [ ]:
## =====================================================================
## Reproducible ESN Quantile Forecast (Simulation / DLM-produced series)
## Single runnable chunk — uses exDQLM functions (qdesn_fit_vb, forecast_paths, synthesis)
## =====================================================================

## -----------------------------
## 0) Setup (packages & exDQLM)
## -----------------------------
req_pkgs <- c(
  "devtools","ggplot2","dplyr","tidyr","tibble","scales",
  "MASS","numDeriv","matrixStats","purrr","readr","patchwork"
)
need <- setdiff(req_pkgs, rownames(installed.packages()))
if (length(need)) install.packages(need, dependencies = TRUE)
invisible(lapply(req_pkgs, require, character.only = TRUE))

## --- repo root (works when running from notebooks/) ---
repo_root <- tryCatch(
  normalizePath(system("git rev-parse --show-toplevel", intern = TRUE)),
  error = function(...) normalizePath("..", mustWork = TRUE)  # fallback: parent of notebooks/
)

# Load your local package so qdesn_fit_vb(), forecast_paths.qdesn_fit(), etc. are available
devtools::load_all(repo_root)
set.seed(12345)  # full determinism
## -------------------------------------------
## 1) Paths & Data Files
## -------------------------------------------
base_dir  <- file.path(repo_root, "results", "sim_suite_dlm", "series", "dlm_constV_smallW")
file_long <- file.path(base_dir, "series_long.csv")  # columns: t, p, q, y, mu
stopifnot(file.exists(file_long))

## ---------------------------------------------------------
## 2) Config — Model, Sampling, Plotting, Synthesis, Output
## ---------------------------------------------------------
# Targeted quantiles (fit one model per p0)
p_vec <- c(0.05, 0.50, 0.95)

# ESN/DESN spec (fixed; same reservoir across quantiles)
desn_args <- list(
  D = 1L,
  n = c(600L),
  n_tilde = integer(0),
  m = 50L,
  alpha = 0.6,
  rho = c(0.90),
  act_f = "tanh",
  act_k = "identity",
  pi_w  = 0.05,
  pi_in = 1.00,
  washout = 500L,
  add_bias = TRUE,
  seed = 42       # ensures identical reservoir across quantiles
)

# VB options for exAL static readout (base for p = 0.50)
vb_args_base <- list(max_iter = 150, tol = 1e-4, n_samp_xi = 500, verbose = TRUE)
# utils used early
near_equal <- function(x, y, tol = 1e-8) abs(x - y) <= tol

# Choose a tighter ELBO tolerance for extreme quantiles
vb_tol_for <- function(p0) {
  if (near_equal(p0, 0.50)) 1e-4 else 1e-5
}

# Posterior sampling
nd_draws <- 3000L
chunk_sz <- 250L

# Plotting
last_window <- 200L
plot_base   <- 11
col_map <- setNames(c("#ef4444", "#10b981", "#0ea5e9"), as.character(p_vec))

# Synthesis (across quantile models)
synth_isotonic  <- TRUE
synth_rearrange <- TRUE
synth_grid_M    <- 2001L
synth_nsamp     <- 4000L
synth_seed      <- 123

# --- Teacher-forcing + rolling-origin options ---
tf_enable   <- TRUE
tf_first_k  <- desn_args$m   # default warm-start = model lag length m
y_future_obs_explicit <- NULL

# Rolling-origin evaluation (use most recent obs as time moves)
rolling_origin <- TRUE       # <- turn ON rolling origin by default
H_step         <- 1L         # <- at each origin, produce H_step-step-ahead forecasts
# NOTE: With rolling_origin=TRUE we DO NOT teacher-force inside the H_step block
# (to avoid lookahead). At h=1 we use observed lags; at h>=2 we use the model’s
# previous prediction. As the origin rolls forward, newly observed y enter the lags.


# Saving
save_outputs <- TRUE
out_dir <- file.path(base_dir, "fig_esn_quantile_notebook")
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

## -------------------------------
## 3) Helpers (small utilities)
## -------------------------------
# small helpers needed by plotting & summaries
fmt_p <- function(x) sprintf("%.3f", as.numeric(x))

# Interpolate the true quantile at an exact tau for every time t
# Returns a numeric vector aligned to unique sorted t in dat_long
true_q_at_tau <- function(dat_long, tau) {
  stopifnot(is.numeric(tau), length(tau) == 1L)
  dat_long %>%
    dplyr::arrange(t, p) %>%
    dplyr::group_by(t) %>%
    dplyr::summarise(
      q_tau = {
        p_i <- as.numeric(p)
        q_i <- as.numeric(q)
        ord <- order(p_i)
        # rule=2 clamps to boundary; avoids NA outside grid
        approx(x = p_i[ord], y = q_i[ord], xout = tau, method = "linear", rule = 2)$y
      },
      .groups = "drop"
    ) %>%
    dplyr::arrange(t) %>%
    dplyr::pull(q_tau)
}

band_from_draws <- function(mat, level = 0.95) {
  stopifnot(is.matrix(mat))
  probs <- c((1 - level)/2, 0.5, (1 + level)/2)
  qs <- t(apply(mat, 1, stats::quantile, probs = probs, names = FALSE))
  colnames(qs) <- c("lo","med","hi")
  qs
}

# --- Consistent plot theme and compact captions --------------------------------
theme_exdqlm <- function(base_size = plot_base,
                         title_scale = 0.95,
                         subtitle_scale = 0.9,
                         caption_scale = 0.85) {
  ggplot2::theme_minimal(base_size = base_size) +
    ggplot2::theme(
      panel.grid.minor = ggplot2::element_blank(),
      legend.position  = "right",
      plot.title       = ggplot2::element_text(face = "bold", size = ggplot2::rel(title_scale)),
      plot.subtitle    = ggplot2::element_text(color = "grey30", size = ggplot2::rel(subtitle_scale)),
      plot.caption     = ggplot2::element_text(color = "grey40", size = ggplot2::rel(caption_scale))
    )
}

subtitle_ro <- function() {
  if (isTRUE(get0("rolling_origin", ifnotfound = FALSE))) {
    sprintf("rolling origin (H_step = %d)", get0("H_step", ifnotfound = NA_integer_))
  } else {
    "single origin"
  }
}

caption_exdqlm <- function(window) {
  gv <- function(nm, default = NA) {
    val <- get0(nm, envir = .GlobalEnv, inherits = TRUE, ifnotfound = default)
    if (is.null(val)) default else val
  }
  parts <- c(
    sprintf("window: last %d steps", as.integer(window)),
    sprintf("ndraws: %d", as.integer(gv("nd_draws", NA_integer_))),
    if (isTRUE(gv("rolling_origin", FALSE)))
      sprintf("rolling-origin: H_step=%s", as.character(gv("H_step", NA))),
    if (!is.na(gv("synth_isotonic", NA)) && !is.na(gv("synth_rearrange", NA)))
      sprintf("synthesis: isotonic=%s, rearrange=%s",
              as.character(gv("synth_isotonic")), as.character(gv("synth_rearrange")))
  )
  paste(parts[!is.na(parts) & nzchar(parts)], collapse = " • ")
}


# --- 1) Forecast μ̂ band vs TRUE q_p & y ---------------------------------------
# μ̂ ±95% credible band vs true q_p and y
# scope: "Train" or "Forecast"
plot_mu_band <- function(df, p0, scope = "Forecast", window = last_window) {
  i2 <- max(df$h); i1 <- max(1L, i2 - window + 1L)
  d  <- dplyr::filter(df, dplyr::between(h, i1, i2))
  # Coverage of TRUE quantile within μ̂ credible band (not y)
  coverage <- mean(d$q_true >= d$lo & d$q_true <= d$hi, na.rm = TRUE)

  ggplot2::ggplot(d, ggplot2::aes(x = h)) +
    theme_exdqlm() +
    ggplot2::labs(
      title    = sprintf("%s: μ̂ ±95%% vs true qₚ (p = %s)", scope, scales::percent(p0, 1)),
      subtitle = sprintf("q_true-in-band = %s", scales::percent(coverage, 0.1)),
      caption  = caption_exdqlm(window),
      x = "time index", y = "value"
    ) +
    ggplot2::geom_ribbon(
      ggplot2::aes(ymin = lo, ymax = hi),
      fill = scales::alpha(col_map[as.character(p0)], 0.22), colour = NA
    ) +
    ggplot2::geom_line(ggplot2::aes(y = mu,     colour = "mu"),   linewidth = 0.95) +
    ggplot2::geom_line(ggplot2::aes(y = q_true, colour = "true"), linewidth = 0.9, linetype = 2) +
    ggplot2::geom_line(ggplot2::aes(y = y,      colour = "data"), linewidth = 0.6, alpha = 0.9) +
    ggplot2::scale_color_manual(
      name   = "",
      values = c(mu   = col_map[as.character(p0)],
                 true = "#7c3aed",
                 data = "#6b7280"),
      breaks = c("mu","true","data"),
      labels = c("μ̂", "true qₚ", "observed y")
    ) +
    ggplot2::scale_x_continuous(expand = c(0, 0)) +
    ggplot2::guides(colour = ggplot2::guide_legend(override.aes = list(linewidth = 1.1, alpha = 1)))
}


# --- 2) Predictive q_p (from draws) vs TRUE q_p --------------------------------
# Empirical quantile q̂_p (from posterior draws) vs true q_p (no bands)
plot_empirical_quantile <- function(df, p0, scope = "Forecast", window = last_window) {
  i2 <- max(df$h); i1 <- max(1L, i2 - window + 1L)
  d  <- dplyr::filter(df, dplyr::between(h, i1, i2))
  mae <- mean(abs(d$q_pred - d$q_true), na.rm = TRUE)

  ggplot2::ggplot(d, ggplot2::aes(x = h)) +
    theme_exdqlm() +
    ggplot2::labs(
      title    = sprintf("%s: q̂ₚ vs true qₚ (p = %s)", scope, scales::percent(p0, 1)),
      subtitle = sprintf("MAE(q̂ₚ, q_true) = %.3f", mae),
      caption  = caption_exdqlm(window),
      x = "time index", y = "value"
    ) +
    ggplot2::geom_line(ggplot2::aes(y = q_pred, colour = "pred"), linewidth = 0.95) +
    ggplot2::geom_line(ggplot2::aes(y = q_true, colour = "true"), linewidth = 0.9, linetype = 2) +
    ggplot2::geom_line(ggplot2::aes(y = y,      colour = "data"), linewidth = 0.6, alpha = 0.85) +
    ggplot2::scale_color_manual(
      name   = "",
      values = c(pred = col_map[as.character(p0)],
                 true = "#7c3aed",
                 data = "#6b7280"),
      breaks = c("pred","true","data"),
      labels = c("q̂ₚ", "true qₚ", "observed y")
    ) +
    ggplot2::scale_x_continuous(expand = c(0, 0)) +
    ggplot2::guides(colour = ggplot2::guide_legend(override.aes = list(linewidth = 1.1, alpha = 1)))
}

# --- 3) Synthesized q_p vs TRUE q_p --------------------------------------------
# Synthesized q_p vs true q_p (no bands)
plot_synth_q_vs_true <- function(df_s, tau, scope = "Forecast", window = last_window) {
  tau_lab <- fmt_p(tau)
  c_true  <- paste0("true_q_",  tau_lab)
  c_synth <- paste0("synth_q_", tau_lab)

  i2 <- max(df_s$h); i1 <- max(1L, i2 - window + 1L)
  d  <- dplyr::filter(df_s, dplyr::between(h, i1, i2))
  mae <- mean(abs(d[[c_synth]] - d[[c_true]]), na.rm = TRUE)

  ggplot2::ggplot(d, ggplot2::aes(x = h)) +
    theme_exdqlm() +
    ggplot2::labs(
      title    = sprintf("%s: synthesized qₚ vs true qₚ (p = %s)", scope, scales::percent(as.numeric(tau), 1)),
      subtitle = sprintf("MAE(synth qₚ, q_true) = %.3f", mae),
      caption  = caption_exdqlm(window),
      x = "time index", y = "value"
    ) +
    ggplot2::geom_line(ggplot2::aes(y = .data[[c_synth]], colour = "synth"), linewidth = 0.95) +
    ggplot2::geom_line(ggplot2::aes(y = .data[[c_true]],  colour = "true"),  linewidth = 0.9, linetype = 2) +
    ggplot2::geom_line(ggplot2::aes(y = y,                 colour = "data"),  linewidth = 0.6, alpha = 0.85) +
    ggplot2::scale_color_manual(
      name   = "",
      values = c(synth = "#0ea5e9", true = "#7c3aed", data = "#6b7280"),
      breaks = c("synth","true","data"),
      labels = c("synth qₚ", "true qₚ", "observed y")
    ) +
    ggplot2::scale_x_continuous(expand = c(0, 0)) +
    ggplot2::guides(colour = ggplot2::guide_legend(override.aes = list(linewidth = 1.1, alpha = 1)))
}

# --- 4) Observed y with synthesized 95% predictive band -------------------------
# Observed y with synthesized 95% predictive band (+ optional median)
plot_synth_predictive_band <- function(synth_draws, y_vec, scope = "Forecast",
                                       window = 50L, fill_col = "#0ea5e9",
                                       show_median = TRUE) {
  stopifnot(is.matrix(synth_draws), length(y_vec) == nrow(synth_draws))
  T_h <- nrow(synth_draws)
  i2 <- T_h; i1 <- max(1L, i2 - as.integer(window) + 1L)

  q_mat <- t(apply(synth_draws, 1L, stats::quantile,
                   probs = c(0.025, 0.50, 0.975), names = FALSE))
  colnames(q_mat) <- c("q05","q50","q95")

  df <- tibble::tibble(
    h   = seq_len(T_h),
    y   = y_vec,
    q05 = q_mat[, "q05"],
    q50 = q_mat[, "q50"],
    q95 = q_mat[, "q95"]
  ) |>
    dplyr::filter(dplyr::between(h, i1, i2))

  coverage <- mean(df$y >= df$q05 & df$y <= df$q95, na.rm = TRUE)
  mean_w   <- mean(df$q95 - df$q05, na.rm = TRUE)

  ggplot2::ggplot(df, ggplot2::aes(x = h)) +
    theme_exdqlm() +
    ggplot2::labs(
      title    = sprintf("%s: synthesized 95%% predictive band", scope),
      subtitle = paste(sprintf("predictive coverage = %s", scales::percent(coverage, 0.1)),
                       sprintf("mean width = %.3f", mean_w), sep = " • "),
      caption  = caption_exdqlm(window),
      x = "time index", y = "value"
    ) +
    ggplot2::geom_ribbon(
      ggplot2::aes(ymin = q05, ymax = q95),
      fill   = scales::alpha(fill_col, 0.22),
      colour = NA
    ) +
    { if (isTRUE(show_median))
        ggplot2::geom_line(ggplot2::aes(y = q50, colour = "median"), linewidth = 0.8)
      else ggplot2::geom_blank()
    } +
    ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"), linewidth = 0.75) +
    ggplot2::scale_color_manual(
      name = "", breaks = c("data","median"),
      values = c(data = "#6b7280", median = fill_col),
      labels = c("observed y","predictive median")
    ) +
    ggplot2::scale_x_continuous(expand = c(0, 0)) +
    ggplot2::guides(colour = ggplot2::guide_legend(override.aes = list(linewidth = 1.1, alpha = 1)))
}


## ------------------------------------------------
## 4) Load data + limit length + Train/Test split
## ------------------------------------------------
dat_long <- read.csv(file_long) |>
  tibble::as_tibble() |>
  dplyr::mutate(
    t = as.integer(t),
    p = as.numeric(p),
    q = as.numeric(q),
    y = as.numeric(y),
    mu = as.numeric(mu)
  ) |>
  dplyr::arrange(t, p)

# Observed y by time
y_full_all <- dat_long |>
  dplyr::distinct(t, y) |>
  dplyr::arrange(t)

T_full <- nrow(y_full_all)
T_use  <- min(T_full, 5000L)  # cap to first x
y_full <- y_full_all[seq_len(T_use), , drop = FALSE]

# 90/10 split by time order
n_train    <- max(1L, floor(0.9 * T_use))
H_forecast <- T_use - n_train
stopifnot(n_train > 0, H_forecast > 0, n_train + H_forecast == T_use)

idx_tr <- 1:n_train
idx_fc <- (n_train + 1):T_use

y_train    <- y_full$y[idx_tr]
y_forecast <- y_full$y[idx_fc]

# Build y_future_obs vector for forecast_paths() based on config
y_future_obs_fc <- {
  if (!isTRUE(tf_enable)) {
    rep(NA_real_, H_forecast)
  } else if (!is.null(y_future_obs_explicit)) {
    stopifnot(is.numeric(y_future_obs_explicit), length(y_future_obs_explicit) == H_forecast)
    as.numeric(y_future_obs_explicit)
  } else if (is.null(tf_first_k)) {
    # teacher-force ALL horizons
    as.numeric(y_forecast)
  } else {
    # teacher-force ONLY first k horizons
    k <- max(0L, min(as.integer(tf_first_k), H_forecast))
    vec <- rep(NA_real_, H_forecast)
    if (k > 0L) vec[seq_len(k)] <- y_forecast[seq_len(k)]
    vec
  }
}

## ---------------------------------------------------------
## 5) Fit (train) and Forecast (holdout) per targeted p0
## ---------------------------------------------------------
fit_and_forecast_p <- function(p0) {
  vb_args_p <- vb_args_base
  vb_args_p$tol <- vb_tol_for(p0)

  # Fit on TRAIN with fixed reservoir spec
  fit_tr <- do.call(
    qdesn_fit_vb,
    c(list(y = y_train, p0 = p0, vb_args = vb_args_p), desn_args)
  )

  m_lag <- as.integer(fit_tr$meta$m)

  if (!isTRUE(rolling_origin)) {
    # (A) Single-origin
    y_hist <- if (m_lag > 0L) tail(y_train, m_lag) else numeric(0)
    fc <- forecast_paths.qdesn_fit(
      object       = fit_tr,
      H            = H_forecast,
      nd           = nd_draws,
      method       = "recursive",
      y_hist       = y_hist,
      y_future_obs = y_future_obs_fc,
      chunk        = chunk_sz
    )
    yrep_fc     <- fc$yrep
    mu_draws_fc <- fc$mu_draws
  } else {
    # (B) Rolling-origin
    yrep_ro_list     <- replicate(H_step, matrix(NA_real_, H_forecast, nd_draws), simplify = FALSE)
    mu_draws_ro_list <- replicate(H_step, matrix(NA_real_, H_forecast, nd_draws), simplify = FALSE)

    for (o in 0:(H_forecast - 1L)) {
      H_o <- min(H_step, H_forecast - o)
      idx_end   <- n_train + o
      y_hist_o  <- if (m_lag > 0L) tail(y_full$y[seq_len(idx_end)], m_lag) else numeric(0)
      yfo <- rep(NA_real_, H_o)

      if (o == 0L && isTRUE(tf_enable) && as.integer(tf_first_k) > 0L) {
        message("rolling_origin=TRUE: ignoring tf_first_k inside blocks to avoid lookahead; only observed lags at each origin are used.")
      }

      fc_o <- forecast_paths.qdesn_fit(
        object       = fit_tr,
        H            = H_o,
        nd           = nd_draws,
        method       = "recursive",
        y_hist       = y_hist_o,
        y_future_obs = yfo,
        chunk        = chunk_sz
      )
      for (s in 1:H_o) {
        yrep_ro_list[[s]][o + 1L, ]     <- fc_o$yrep[s, ]
        mu_draws_ro_list[[s]][o + 1L, ] <- fc_o$mu_draws[s, ]
      }
    }
    yrep_fc     <- yrep_ro_list[[1L]]
    mu_draws_fc <- mu_draws_ro_list[[1L]]
    if (H_step >= 2L) {
      yrep_fc_h2     <- yrep_ro_list[[2L]]
      mu_draws_fc_h2 <- mu_draws_ro_list[[2L]]
    }
  }

  # Summaries for plotting (Forecast)
  q_pred_fc <- apply(yrep_fc, 1, stats::quantile, probs = p0, names = FALSE)
  mu_qs_fc  <- band_from_draws(mu_draws_fc, level = 0.95)

  # TRUE q_p for Forecast via interpolation
  q_true_full <- true_q_at_tau(dat_long, tau = p0)
  stopifnot(length(q_true_full) >= T_use)
  q_true_fc <- q_true_full[idx_fc]

  df_mu_fc <- tibble::tibble(
    h         = seq_len(H_forecast),
    p0        = p0,
    mu        = mu_qs_fc[, "med"],
    lo        = mu_qs_fc[, "lo"],
    hi        = mu_qs_fc[, "hi"],
    q_true    = q_true_fc,
    y         = y_forecast
  )

  df_pred_fc <- tibble::tibble(
    h         = seq_len(H_forecast),
    p0        = p0,
    q_pred    = q_pred_fc,
    q_true    = q_true_fc,
    y         = y_forecast
  )

  out <- list(
    fit_train   = fit_tr,
    yrep_fc     = yrep_fc,
    mu_draws_fc = mu_draws_fc,
    df_mu_fc    = df_mu_fc,
    df_pred_fc  = df_pred_fc
  )
  if (exists("yrep_fc_h2")) {
    out$yrep_fc_h2     <- yrep_fc_h2
    out$mu_draws_fc_h2 <- mu_draws_fc_h2
  }
  out
}


## ---------------------------------------------------------
## 6) Run fits for p ∈ p_vec and produce per-p forecast plots
## ---------------------------------------------------------
fits_fc <- lapply(p_vec, fit_and_forecast_p)
names(fits_fc) <- paste0("p=", p_vec)

for (k in seq_along(p_vec)) {
  p0 <- p_vec[k]
  g1 <- plot_mu_band(fits_fc[[k]]$df_mu_fc, p0, scope = "Forecast", window = last_window)
  g2 <- plot_empirical_quantile(fits_fc[[k]]$df_pred_fc, p0, scope = "Forecast", window = last_window)
  print(g1); print(g2)

  if (isTRUE(save_outputs)) {
    ggsave(file.path(out_dir, sprintf("forecast_mu_band_p=%s.png", as.character(p0))),
           g1, width = 9, height = 4.8, dpi = 150)
    ggsave(file.path(out_dir, sprintf("forecast_emp_q_vs_true_p=%s.png", as.character(p0))),
           g2, width = 9, height = 4.8, dpi = 150)
  }
}


## ---------------------------------------------------------
## 6.1) ELBO trace plots for all quantile models (skip first k)
## ---------------------------------------------------------
k_burn <- 20  # ignore first k iterations

# build a tidy data frame of ELBO traces across p in p_vec
elbo_df <- dplyr::bind_rows(lapply(seq_along(fits_fc), function(i) {
  tr <- fits_fc[[i]]$fit_train$fit$misc$elbo
  if (is.null(tr) || !length(tr)) return(tibble::tibble())
  tibble::tibble(
    p0   = p_vec[i],
    iter = seq_along(tr),
    elbo = as.numeric(tr)
  )
}))

if (!nrow(elbo_df)) {
  message("No ELBO traces found. (Check fits_fc[[i]]$fit_train$fit$misc$elbo)")
} else {
  elbo_df <- elbo_df |>
  dplyr::filter(iter > k_burn) |>
  dplyr::mutate(p0_chr = factor(as.character(p0), levels = as.character(p_vec)))

  g_elbo <- ggplot2::ggplot(elbo_df, ggplot2::aes(x = iter, y = elbo, colour = p0_chr)) +
    theme_exdqlm() +
    ggplot2::labs(
      x = "VB iteration", y = "ELBO", colour = "p0",
      title = "ELBO traces across quantile models",
      subtitle = sprintf("First k=%d iterations omitted", k_burn),
      caption = sprintf("nd: %d", nd_draws)
    ) +
    ggplot2::geom_line(linewidth = 0.8, alpha = 0.95) +
    ggplot2::scale_color_manual(values = col_map) +
    ggplot2::scale_x_continuous(expand = c(0, 0))


  print(g_elbo)
  if (isTRUE(save_outputs)) {
    ggplot2::ggsave(file.path(out_dir, sprintf("elbo_traces_skip_k=%d.png", k_burn)),
                    g_elbo, width = 9, height = 4.8, dpi = 150)
  }
}

## ---------------------------------------------------------
## 6B) In-sample fit checks (train, post-washout)
## ---------------------------------------------------------
train_window <- 200L  # window to visualize from the end of the train chunk

for (k in seq_along(p_vec)) {
  p0   <- p_vec[k]
  obj  <- fits_fc[[k]]$fit_train
  keep <- obj$meta$keep_idx              # indices inside y_train used in the fit
  y_tr_keep <- y_train[keep]
  q_true_full <- true_q_at_tau(dat_long, tau = p0)
  q_true_tr   <- q_true_full[keep]


  # posterior predictive on the training design (aligned to keep_idx)
  pp_tr <- posterior_predict.qdesn_fit(obj, nd = nd_draws, chunk = chunk_sz)
  yrep_tr     <- pp_tr$yrep          # (n_train_kept × nd)
  mu_draws_tr <- pp_tr$mu_draws      # (n_train_kept × nd)

  # summaries
  q_pred_tr <- apply(yrep_tr, 1, stats::quantile, probs = p0, names = FALSE)
  mu_qs_tr  <- band_from_draws(mu_draws_tr, level = 0.95)

  # tidy frames (h = 1..n_train_kept)
  df_mu_tr <- tibble::tibble(
    h      = seq_along(keep),
    p0     = p0,
    mu     = mu_qs_tr[, "med"],
    lo     = mu_qs_tr[, "lo"],
    hi     = mu_qs_tr[, "hi"],
    q_true = q_true_tr,
    y      = y_tr_keep
  )
  df_pred_tr <- tibble::tibble(
    h      = seq_along(keep),
    p0     = p0,
    q_pred = q_pred_tr,
    q_true = q_true_tr,
    y      = y_tr_keep
  )

  # stash for later synthesis on train
  fits_fc[[k]]$yrep_tr      <- yrep_tr
  fits_fc[[k]]$mu_draws_tr  <- mu_draws_tr
  fits_fc[[k]]$df_mu_tr     <- df_mu_tr
  fits_fc[[k]]$df_pred_tr   <- df_pred_tr

  # plots (reuse your forecast plotters; just tweak titles)
  g1_tr <- plot_mu_band(df_mu_tr, p0, scope = "Train", window = train_window)
  g2_tr <- plot_empirical_quantile(df_pred_tr, p0, scope = "Train", window = train_window)

  print(g1_tr); print(g2_tr)
  if (isTRUE(save_outputs)) {
    ggsave(file.path(out_dir, sprintf("train_mu_band_p=%s.png", as.character(p0))),
           g1_tr, width = 9, height = 4.8, dpi = 150)
    ggsave(file.path(out_dir, sprintf("train_emp_q_vs_true_p=%s.png", as.character(p0))),
       g2_tr, width = 9, height = 4.8, dpi = 150)
  }
}


## ---------------------------------------------------------
## 7) Synthesize across per-p models on the forecast window
## ---------------------------------------------------------
# Collect forecast predictive draws (H × nd) for each fitted quantile model
draws_list_fc <- lapply(fits_fc, function(obj) obj$yrep_fc)

synth_fc <- exdqlm_synthesize_from_draws(
  draws_list      = draws_list_fc,
  p               = p_vec,
  enforce_isotonic = synth_isotonic,
  rearrange        = synth_rearrange,
  grid_M           = synth_grid_M,
  n_samp           = synth_nsamp,
  seed             = synth_seed,
  T_expected       = H_forecast
)
# synth_fc$draws : (H_forecast × synth_nsamp)

# Build synthesized quantiles over horizon (named by target taus)
p_comp <- c(0.05, 0.50, 0.95)
synth_cols_fc <- lapply(p_comp, function(tau)
  apply(synth_fc$draws, 1L, stats::quantile, probs = tau, names = FALSE))
names(synth_cols_fc) <- paste0("synth_q_", fmt_p(p_comp))
synth_q_fc <- tibble::as_tibble(synth_cols_fc)

# Build TRUE q_p for the same target taus via interpolation
true_cols <- setNames(vector("list", length(p_comp)), paste0("true_q_", fmt_p(p_comp)))
for (i in seq_along(p_comp)) {
  q_true_full_i <- true_q_at_tau(dat_long, tau = p_comp[i])
  stopifnot(length(q_true_full_i) >= T_use)
  true_cols[[i]] <- q_true_full_i[idx_fc]
}
true_q_fc <- tibble::as_tibble(true_cols)

# Assemble comparison frame for forecast horizon
compare_fc <- tibble::tibble(
  h = seq_len(H_forecast),
  y = y_forecast
) |>
  dplyr::bind_cols(true_q_fc) |>
  dplyr::bind_cols(synth_q_fc)

# Plots for synthesized vs true at selected taus
plots_synth_fc <- lapply(p_comp, function(tau) {
  plot_synth_q_vs_true(compare_fc, tau, scope = "Forecast", window = last_window)
})
for (j in seq_along(plots_synth_fc)) {
  print(plots_synth_fc[[j]])
  if (isTRUE(save_outputs)) {
    ggsave(file.path(out_dir, sprintf("forecast_synth_vs_true_p=%s.png", fmt_p(p_comp[j]))),
           plots_synth_fc[[j]], width = 9, height = 4.8, dpi = 150)
  }
}

## ---------------------------------------------------------
## 7B) Synthesis on the training (post-washout) chunk
## ---------------------------------------------------------
draws_list_tr <- lapply(fits_fc, function(obj) obj$yrep_tr)
T_train_keep  <- nrow(draws_list_tr[[1]])  # same across p if m/washout identical
keep_train <- fits_fc[[1]]$fit_train$meta$keep_idx

synth_tr <- exdqlm_synthesize_from_draws(
  draws_list      = draws_list_tr,
  p               = p_vec,
  enforce_isotonic = synth_isotonic,
  rearrange        = synth_rearrange,
  grid_M           = synth_grid_M,
  n_samp           = synth_nsamp,
  seed             = synth_seed,
  T_expected       = T_train_keep
)

# synthesized quantiles on train
p_comp <- c(0.05, 0.50, 0.95)
synth_cols_tr <- lapply(p_comp, function(tau)
  apply(synth_tr$draws, 1L, stats::quantile, probs = tau, names = FALSE))
names(synth_cols_tr) <- paste0("synth_q_", fmt_p(p_comp))
synth_q_tr <- tibble::as_tibble(synth_cols_tr)

# TRUE q_p for train kept indices
true_cols_tr <- setNames(vector("list", length(p_comp)), paste0("true_q_", fmt_p(p_comp)))
for (i in seq_along(p_comp)) {
  q_true_full_i <- true_q_at_tau(dat_long, tau = p_comp[i])
  true_cols_tr[[i]] <- q_true_full_i[keep_train]
}
true_q_tr <- tibble::as_tibble(true_cols_tr)


# assemble comparison frame (train)
compare_tr <- tibble::tibble(
  h = seq_len(T_train_keep),
  y = y_train[keep_train]
) |>
  dplyr::bind_cols(true_q_tr) |>
  dplyr::bind_cols(synth_q_tr)

# plots on train
plots_synth_tr <- lapply(p_comp, function(tau) {
  plot_synth_q_vs_true(compare_tr, tau, scope = "Train", window = train_window)
})

for (j in seq_along(plots_synth_tr)) {
  print(plots_synth_tr[[j]])
  if (isTRUE(save_outputs)) {
    ggsave(file.path(out_dir, sprintf("train_synth_vs_true_p=%s.png", fmt_p(p_comp[j]))),
           plots_synth_tr[[j]], width = 9, height = 4.8, dpi = 150)
  }
}

# optional: observed y with synthesized 95% band (train)
g_band_tr <- plot_synth_predictive_band(
  synth_draws = synth_tr$draws,
  y_vec = y_train[keep_train],
  scope       = "Train",
  window      = train_window,
  fill_col    = "#0ea5e9",
  show_median = TRUE
)

print(g_band_tr)
if (isTRUE(save_outputs)) {
  ggsave(file.path(out_dir, "train_obs_with_95_band.png"),
         g_band_tr, width = 9, height = 4.8, dpi = 150)
}

## ---------------------------------------------------------
## 8) Observed y with synthesized 95% predictive band (forecast window)
## ---------------------------------------------------------
g_band_fc <- plot_synth_predictive_band(
  synth_draws = synth_fc$draws,
  y_vec       = y_forecast,
  scope       = "Forecast",
  window      = last_window,
  fill_col    = "#3B82F6",
  show_median = TRUE
)

print(g_band_fc)

if (isTRUE(save_outputs)) {
  ggsave(file.path(out_dir, "forecast_obs_with_95_band.png"),
         g_band_fc, width = 9, height = 4.8, dpi = 150)
}

## ---------------------------------------------------------
## 9) Optional: save objects (fits & synthesis) for inspection
## ---------------------------------------------------------
saveRDS(
  list(
    fits_fc = fits_fc, synth_fc = synth_fc, compare_fc = compare_fc,
    cfg = list(
      p_vec = p_vec, desn_args = desn_args,
      vb_args_base = vb_args_base,
      vb_tol_by_p = setNames(sapply(p_vec, vb_tol_for), as.character(p_vec)),  # helpful breadcrumb
      nd_draws = nd_draws, chunk_sz = chunk_sz, last_window = last_window,
      teacher_forcing = list(enable = tf_enable, first_k = tf_first_k,
                             explicit = y_future_obs_explicit,
                             y_future_obs_fc = y_future_obs_fc),
      synth = list(isotonic = synth_isotonic, rearrange = synth_rearrange,
                   grid_M = synth_grid_M, n_samp = synth_nsamp, seed = synth_seed),
      split = list(T_use = T_use, n_train = n_train, H_forecast = H_forecast)
    )
  ),
  file.path(out_dir, "forecast_objects.rds")
)

## ================================================================
## 11) Calibration diagnostics (μ and q̂ₚ): global + rolling coverage
## ================================================================

# --- helpers (re-use + a small vectorized wrapper) ----------------
wilson_ci <- function(k, n, conf = 0.95) {
  if (n <= 0) return(c(NA_real_, NA_real_))
  z <- stats::qnorm(0.5 + conf/2)
  p <- k / n
  den <- 1 + z^2 / n
  cen <- (p + z^2/(2*n)) / den
  rad <- z * sqrt(p*(1-p)/n + z^2/(4*n^2)) / den
  c(max(0, cen - rad), min(1, cen + rad))
}

pinball_loss <- function(y, qhat, p) {
  e <- y - qhat
  (p - (e < 0)) * e
}

roll_mean <- function(x, W) {
  if (W <= 1) return(x)
  as.numeric(stats::filter(x, rep(1 / W, W), sides = 1))
}

# --- build long frames aligned in "time" for train and forecast ---
# We add t_aligned so we can mix train + forecast on a single x-axis if desired.

# μ_t frames (train + forecast)
mu_tr_long <- dplyr::bind_rows(purrr::compact(lapply(seq_along(p_vec), function(i) {
  d <- fits_fc[[i]]$df_mu_tr
  if (is.null(d) || !nrow(d)) return(NULL)
  keep <- fits_fc[[i]]$fit_train$meta$keep_idx
  d %>% dplyr::mutate(scope="train", p_chr=sprintf("%.2f", p_vec[i]),
                      t_aligned=keep, mu_hat=mu)
})))


mu_fc_long <- dplyr::bind_rows(lapply(seq_along(p_vec), function(i) {
  d <- fits_fc[[i]]$df_mu_fc
  d |>
    dplyr::mutate(scope = "forecast",
                  p_chr = sprintf("%.2f", p_vec[i]),
                  t_aligned = n_train + h,
                  mu_hat = mu)
}))

mu_long <- dplyr::bind_rows(mu_tr_long, mu_fc_long)

# q̂_p frames (train + forecast) from posterior draws
q_tr_long <- dplyr::bind_rows(lapply(seq_along(p_vec), function(i) {
  d <- fits_fc[[i]]$df_pred_tr
  keep <- fits_fc[[i]]$fit_train$meta$keep_idx
  d |>
    dplyr::mutate(scope = "train",
                  p_chr = sprintf("%.2f", p_vec[i]),
                  t_aligned = keep,
                  qhat = q_pred)
}))

q_fc_long <- dplyr::bind_rows(lapply(seq_along(p_vec), function(i) {
  d <- fits_fc[[i]]$df_pred_fc
  d |>
    dplyr::mutate(scope = "forecast",
                  p_chr = sprintf("%.2f", p_vec[i]),
                  t_aligned = n_train + h,
                  qhat = q_pred)
}))

q_long <- dplyr::bind_rows(q_tr_long, q_fc_long)

# Synthesized q_p long frames (Train + Forecast) for rolling coverage
# Need train 'keep' and p_comp = c(0.05, 0.50, 0.95) already defined above
qsynth_tr_long <- dplyr::bind_rows(lapply(p_comp, function(tau) {
  tibble::tibble(
    scope     = "train",
    p0        = tau,
    p_chr     = sprintf("%.2f", tau),
    t_aligned = keep_train,
    q_synth   = synth_q_tr[[paste0("synth_q_", fmt_p(tau))]],
    y         = y_train[keep_train]
  )
}))


qsynth_fc_long <- dplyr::bind_rows(lapply(p_comp, function(tau) {
  tibble::tibble(
    scope     = "forecast",
    p0        = tau,
    p_chr     = sprintf("%.2f", tau),
    t_aligned = n_train + seq_len(H_forecast),
    q_synth   = synth_q_fc[[paste0("synth_q_", fmt_p(tau))]],
    y         = y_forecast
  )
}))

qsynth_long <- dplyr::bind_rows(qsynth_tr_long, qsynth_fc_long)

# --- GLOBAL coverage tables ---------------------------------------
summarize_cov_tbl <- function(df, qcol) {
  # df must contain: y, p0, scope, and qcol (either "mu_hat" or "qhat")
  stopifnot(all(c("y","p0","scope", qcol) %in% names(df)))
  df |>
    dplyr::filter(is.finite(.data$y), is.finite(.data[[qcol]])) |>
    dplyr::group_by(scope, p0) |>
    dplyr::summarise(
      N = dplyr::n(),
      k = sum(.data$y <= .data[[qcol]], na.rm = TRUE),
      coverage = ifelse(N > 0, k / N, NA_real_),
      cov_lo95 = wilson_ci(k, N)[1],
      cov_hi95 = wilson_ci(k, N)[2],
      pinball  = mean(pinball_loss(.data$y, .data[[qcol]], dplyr::first(p0)), na.rm = TRUE),
      .groups = "drop"
    ) |>
    dplyr::arrange(scope, p0)
}

stopifnot(nrow(mu_long) > 0, nrow(q_long) > 0)

cov_mu_tbl   <- summarize_cov_tbl(mu_long, "mu_hat")
cov_qhat_tbl <- summarize_cov_tbl(q_long, "qhat")

print(cov_mu_tbl)
print(cov_qhat_tbl)

# --- Rolling coverage plots ---------------------------------------
cov_window <- 365L     # trailing window length for rolling coverage
show_last  <- 300L     # only show last X points for clarity

# A single plotting function that works for μ or q̂_p
plot_rolling_cov <- function(df_long, qcol, window = cov_window, show_last = 500L,
                             title_left = "Rolling empirical coverage") {
  stopifnot(all(c("t_aligned","y","p0","p_chr") %in% names(df_long)))
  d <- df_long |>
    dplyr::mutate(ind = as.integer(.data$y <= .data[[qcol]])) |>
    dplyr::arrange(scope, p_chr, t_aligned) |>
    dplyr::group_by(scope, p_chr) |>
    dplyr::mutate(rcov = roll_mean(ind, window),
                  t_max = max(t_aligned, na.rm = TRUE)) |>
    dplyr::ungroup() |>
    dplyr::filter(t_aligned > (t_max - show_last))

  d <- d |> dplyr::mutate(p_chr = factor(p_chr, levels = sprintf("%.2f", p_vec)))

  bands <- d |>
    dplyr::distinct(scope, p_chr, p0) |>
    dplyr::rowwise() |>
    dplyr::mutate(
      se = sqrt(p0 * (1 - p0) / window),
      lo = p0 - stats::qnorm(0.975) * se,
      hi = p0 + stats::qnorm(0.975) * se
    ) |>
    dplyr::ungroup()

  x_rng <- range(d$t_aligned, na.rm = TRUE)
  bands <- bands |>
    dplyr::mutate(xmin = x_rng[1], xmax = x_rng[2])

  # last points per line and shifted label positions
  last_pts <- d %>%
    dplyr::group_by(scope, p_chr) %>%
    dplyr::slice_tail(n = 1) %>%
    dplyr::ungroup() %>%
    dplyr::mutate(
      x_lab = t_aligned - 0.03 * diff(x_rng),
      y_lab = pmin(pmax(rcov + 0.02, 0), 1)
    )

  ggplot2::ggplot(d, ggplot2::aes(x = t_aligned, y = rcov, colour = p_chr)) +
    theme_exdqlm() +
    ggplot2::labs(
      x = "time index (aligned)",
      y = sprintf("rolling Pr(y ≤ %s)  (W = %d)", if (qcol=="mu_hat") "μ" else "q̂ₚ", window),
      title    = paste0(title_left, if (qcol=="mu_hat") " of μ" else " of q̂ₚ"),
      subtitle = paste(subtitle_ro(),
                       sprintf("Last %d points; shaded: ±1.96√(p(1-p)/W)", show_last),
                       sep = " • ")
    ) +
    ggplot2::geom_hline(data = bands,
                        ggplot2::aes(yintercept = p0, colour = p_chr),
                        linetype = "dashed", linewidth = 0.7, show.legend = FALSE) +
    ggplot2::geom_rect(data = bands,
                       ggplot2::aes(xmin = xmin, xmax = xmax, ymin = lo, ymax = hi, fill = p_chr),
                       inherit.aes = FALSE, alpha = 0.12) +
    ggplot2::geom_line(linewidth = 0.9, na.rm = TRUE) +
    ggplot2::geom_point(data = last_pts, size = 2.4) +
    ggplot2::geom_text(data = last_pts,
                       ggplot2::aes(x = x_lab, y = y_lab, label = sprintf("%.2f", rcov)),
                       size = 3, hjust = 1) +
    ggplot2::scale_color_manual(
      name = "quantile p",
      values = setNames(col_map, sprintf("%.2f", p_vec)),
      labels = function(x) scales::percent(as.numeric(x))
    ) +
    ggplot2::scale_fill_manual(
      name = "quantile p",
      values = setNames(sapply(col_map, scales::alpha, alpha = 0.12), sprintf("%.2f", p_vec)),
      labels = function(x) scales::percent(as.numeric(x))
    ) +
    ggplot2::scale_y_continuous(breaks = seq(0, 1, by = 0.1),
                                labels = scales::percent_format(accuracy = 1)) +
    ggplot2::scale_x_continuous(limits = x_rng, expand = c(0, 0)) +
    ggplot2::coord_cartesian(ylim = c(0, 1), expand = FALSE) +
    ggplot2::guides(colour = ggplot2::guide_legend(override.aes = list(linewidth = 1.2, alpha = 1)))
}


# --- Make the plots (μ and q̂ₚ), train & forecast -----------------
g_cov_mu_train <- plot_rolling_cov(dplyr::filter(mu_long, scope=="train"),
                                   qcol = "mu_hat", window = cov_window, show_last = show_last,
                                   title_left = "Rolling empirical coverage")
g_cov_mu_fore  <- plot_rolling_cov(dplyr::filter(mu_long, scope=="forecast"),
                                   qcol = "mu_hat", window = cov_window, show_last = show_last,
                                   title_left = "Rolling empirical coverage")

g_cov_q_train  <- plot_rolling_cov(dplyr::filter(q_long, scope=="train"),
                                   qcol = "qhat", window = cov_window, show_last = show_last,
                                   title_left = "Rolling empirical coverage")
g_cov_q_fore   <- plot_rolling_cov(dplyr::filter(q_long, scope=="forecast"),
                                   qcol = "qhat", window = cov_window, show_last = show_last,
                                   title_left = "Rolling empirical coverage")

print(g_cov_mu_train); print(g_cov_mu_fore)
print(g_cov_q_train);  print(g_cov_q_fore)

if (isTRUE(save_outputs)) {
  ggplot2::ggsave(file.path(out_dir, sprintf("rolling_cov_mu_train_W=%d.png", cov_window)),
                  g_cov_mu_train, width = 9, height = 4.8, dpi = 150)
  ggplot2::ggsave(file.path(out_dir, sprintf("rolling_cov_mu_forecast_W=%d.png", cov_window)),
                  g_cov_mu_fore, width = 9, height = 4.8, dpi = 150)
  ggplot2::ggsave(file.path(out_dir, sprintf("rolling_cov_qhat_train_W=%d.png", cov_window)),
                  g_cov_q_train, width = 9, height = 4.8, dpi = 150)
  ggplot2::ggsave(file.path(out_dir, sprintf("rolling_cov_qhat_forecast_W=%d.png", cov_window)),
                  g_cov_q_fore, width = 9, height = 4.8, dpi = 150)
}

g_cov_qsynth_train <- plot_rolling_cov(dplyr::rename(qsynth_tr_long, qhat = q_synth),
                                       qcol = "qhat", window = cov_window, show_last = show_last,
                                       title_left = "Rolling empirical coverage")
g_cov_qsynth_fore  <- plot_rolling_cov(dplyr::rename(qsynth_fc_long, qhat = q_synth),
                                       qcol = "qhat", window = cov_window, show_last = show_last,
                                       title_left = "Rolling empirical coverage")

print(g_cov_qsynth_train); print(g_cov_qsynth_fore)
cov_qsynth_tbl <- summarize_cov_tbl(dplyr::rename(qsynth_long, qhat = q_synth), "qhat")

if (isTRUE(save_outputs)) {
  ggplot2::ggsave(file.path(out_dir, sprintf("rolling_cov_qsynth_train_W=%d.png", cov_window)),
                  g_cov_qsynth_train, width = 9, height = 4.8, dpi = 150)
  ggplot2::ggsave(file.path(out_dir, sprintf("rolling_cov_qsynth_forecast_W=%d.png", cov_window)),
                  g_cov_qsynth_fore, width = 9, height = 4.8, dpi = 150)
}

if (isTRUE(save_outputs)) {
  readr::write_csv(cov_mu_tbl,   file.path(out_dir, "calibration_mu_table.csv"))
  readr::write_csv(cov_qhat_tbl, file.path(out_dir, "calibration_qhat_table.csv"))
  readr::write_csv(cov_qsynth_tbl, file.path(out_dir, "calibration_qsynth_table.csv"))
}

## ================================================================
## 10) PIT diagnostics from posterior draws (train & forecast)
## ================================================================
emp_pit_vec <- function(y, yrep_mat) {
  stopifnot(length(y) == nrow(yrep_mat))
  # fraction of draws <= observed y (per time)
  rowMeans(sweep(yrep_mat, 1, y, FUN = "<="), na.rm = TRUE)
}

# Build train/forecast PIT at p=0.50 model (any p works; 0.50 is typical)
i_med <- which.min(abs(p_vec - 0.50))
pit_tr <- emp_pit_vec(y_train[fits_fc[[i_med]]$fit_train$meta$keep_idx],
                      fits_fc[[i_med]]$yrep_tr)
pit_fc <- emp_pit_vec(y_forecast, fits_fc[[i_med]]$yrep_fc)

plot_pit_hist <- function(pit, title) {
  pit <- pit[is.finite(pit)]
  ks  <- suppressWarnings(stats::ks.test(pit, "punif"))  # quick uniformity check
  ggplot2::ggplot(tibble::tibble(pit = pit), ggplot2::aes(x = pit)) +
    theme_exdqlm() +
    ggplot2::geom_histogram(ggplot2::aes(y = after_stat(density)),
                            boundary = 0, bins = 20, color = "white") +
    ggplot2::geom_hline(yintercept = 1, linetype = 2) +
    ggplot2::labs(title = title,
                  subtitle = sprintf("KS p = %.3f", ks$p.value),
                  x = "PIT", y = "density") +
    ggplot2::coord_cartesian(xlim = c(0, 1), ylim = c(0, NA))
}


plot_pit_qq <- function(pit, title) {
  n <- sum(is.finite(pit)); pit_s <- sort(pit[is.finite(pit)])
  u <- stats::ppoints(n)
  ggplot2::ggplot(tibble::tibble(u = u, pit = pit_s), ggplot2::aes(x = u, y = pit)) +
    theme_exdqlm() +
    ggplot2::geom_abline(slope = 1, intercept = 0, linetype = 2) +
    ggplot2::geom_point(alpha = 0.7, size = 1.6) +
    ggplot2::labs(title = title, x = "Uniform(0,1) quantiles", y = "PIT quantiles") +
    ggplot2::coord_cartesian(xlim = c(0,1), ylim = c(0,1))
}

g_pit_tr_hist <- plot_pit_hist(pit_tr, "PIT histogram (train)")
g_pit_fc_hist <- plot_pit_hist(pit_fc, "PIT histogram (forecast)")
g_pit_tr_qq   <- plot_pit_qq(pit_tr,   "PIT QQ (train)")
g_pit_fc_qq   <- plot_pit_qq(pit_fc,   "PIT QQ (forecast)")

g_pit_train    <- g_pit_tr_hist | g_pit_tr_qq
g_pit_forecast <- g_pit_fc_hist | g_pit_fc_qq

if (isTRUE(save_outputs)) {
  ggplot2::ggsave(file.path(out_dir, "pit_train.png"),    g_pit_train,    width = 12, height = 4.5, dpi = 150)
  ggplot2::ggsave(file.path(out_dir, "pit_forecast.png"), g_pit_forecast, width = 12, height = 4.5, dpi = 150)
}

print(g_pit_train)
print(g_pit_forecast)




<!-- # Data - In Sample -->

In [ ]:
# devtools::load_all()

# # --- Timing helper (define once per kernel/session) ---
# if (!exists("timer", mode = "function")) {
#   timer <- function(label, expr) {
#     t <- system.time(res <- eval.parent(substitute(expr)))
#     message(sprintf("[%.2fs] %s", t[["elapsed"]], label))
#     invisible(res)
#   }
# }

# ## ================================================================
# ## 1) Config — simple, single-layer setup
# ## ================================================================
# # Targeted quantiles
# p_vec <- c(0.05, 0.50, 0.95)

# # Reservoir / Q-DESN: single layer, modest size, dense Win, sparse W
# desn_args <- list(
#   D = 1L,
#   n = c(500L),
#   n_tilde = integer(0),     # D=1 -> no reductions
#   m = 3L,                  # lags of the response in the reservoir input
#   alpha = 0.30,             # “minimal ESN” vibe
#   rho = c(0.90),
#   act_f = "tanh", act_k = "identity",
#   pi_w = 0.05,              # sparse internal W
#   pi_in = 1.00,             # dense Win (as in the simple ESN)
#   washout = 500L,
#   add_bias = TRUE,
#   seed = 42
# )

# # VB (readout) fit — simple and consistent
# vb_args <- list(max_iter = 2000, tol = 1e-4, n_samp_xi = 2000, verbose = TRUE)

# # Exogenous covariate lags (keep it small/simple)
# lags_ppt  <- 3L
# lags_soil <- 3L

# # Posterior draws & synthesis (unchanged, but kept smallish for speed)
# nd_draws <- 2000L
# chunk_sz <- 250L
# synth_isotonic  <- TRUE
# synth_rearrange <- TRUE
# synth_grid_M    <- 2001L
# synth_nsamp     <- 3000L
# synth_seed      <- 777

# # Plot window (last N aligned points)
# last_window <- 100L

# # Colors
# col_map <- setNames(c("#ef4444", "#10b981", "#0ea5e9"), as.character(p_vec))

# ## ================================================================
# ## 2) Helpers
# ## ================================================================
# # Build lag matrix (T × L): column j is x_{t-j}
# lag_mat <- function(x, L) {
#   Tlen <- length(x); L <- as.integer(L)
#   if (L <= 0) return(matrix(numeric(0), nrow = Tlen, ncol = 0))
#   M <- matrix(0, nrow = Tlen, ncol = L)
#   for (j in 1:L) M[(j+1):Tlen, j] <- x[1:(Tlen - j)]
#   colnames(M) <- paste0(deparse(substitute(x)), "_l", 1:L)
#   M
# }

# # 95% band + median for μ from qbeta
# mu_band <- function(X, qbeta, level = 0.95) {
#   m <- as.numeric(qbeta$m)
#   V <- as.matrix(qbeta$V)
#   mu <- as.numeric(X %*% m)
#   XV <- X %*% V
#   var <- rowSums(XV * X)
#   se  <- sqrt(pmax(var, 0))
#   z <- qnorm(0.5 + level/2)
#   tibble::tibble(mu = mu, lo = mu - z*se, hi = mu + z*se)
# }

# # window slicer
# slice_window <- function(df, idx_col = "t_aligned", N = 200L) {
#   i2 <- max(df[[idx_col]])
#   i1 <- max(1L, i2 - as.integer(N) + 1L)
#   dplyr::filter(df, dplyr::between(.data[[idx_col]], i1, i2))
# }

# # Plot μ-band vs data (no true quantile available here)
# plot_mu_vs_y <- function(df, p0, window = last_window) {
#   d <- slice_window(df, "t_aligned", window)
#   ggplot2::ggplot(d, ggplot2::aes(x = t_aligned)) +
#     ggplot2::theme_minimal(base_size = 13) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "time (aligned)", y = "USGS (response)",
#       title = sprintf("Q-DESN μ̂ with 95%% credible band — target p = %s",
#                       scales::percent(p0, accuracy = 1)),
#       subtitle = sprintf("Last %d points", nrow(d))
#     ) +
#     ggplot2::geom_ribbon(
#       ggplot2::aes(ymin = lo, ymax = hi),
#       fill = scales::alpha(col_map[as.character(p0)], 0.20),
#       colour = NA
#     ) +
#     ggplot2::geom_line(ggplot2::aes(y = mu, colour = "mu"), linewidth = 0.9) +
#     ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"), linewidth = 0.7) +
#     ggplot2::scale_color_manual(
#       name = "", breaks = c("mu","data"),
#       values = c(mu = col_map[as.character(p0)], data = "#111111"),
#       labels = c("μ̂ (fit)","observed y")
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# # Observed y with synthesized 95% band (2.5–97.5)
# plot_obs_with_95_band <- function(synth_draws, y_vec, window = last_window,
#                                   fill_col = "#3B82F6", show_median = TRUE) {
#   stopifnot(is.matrix(synth_draws), length(y_vec) == nrow(synth_draws))
#   q_mat <- t(apply(synth_draws, 1L, stats::quantile,
#                    probs = c(0.025, 0.50, 0.975), names = FALSE))
#   colnames(q_mat) <- c("q025","q50","q975")
#   df <- tibble::tibble(
#     t_aligned = seq_len(nrow(synth_draws)),
#     y   = y_vec,
#     q05 = q_mat[, "q025"],
#     q50 = q_mat[, "q50"],
#     q95 = q_mat[, "q975"]
#   )
#   d <- slice_window(df, "t_aligned", window)
#   ggplot2::ggplot(d, ggplot2::aes(x = t_aligned)) +
#     ggplot2::theme_minimal(base_size = 13) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "time (aligned)", y = "USGS (response)",
#       title = "Observed y with synthesized 95% predictive band",
#       subtitle = sprintf("Band = [2.5%%, 97.5%%]; last %d points", nrow(d))
#     ) +
#     ggplot2::geom_ribbon(
#       ggplot2::aes(ymin = q05, ymax = q95),
#       fill   = scales::alpha(fill_col, 0.22),
#       colour = NA
#     ) +
#     { if (isTRUE(show_median))
#         ggplot2::geom_line(ggplot2::aes(y = q50, colour = "median"), linewidth = 0.7)
#       else ggplot2::geom_blank()
#     } +
#     ggplot2::geom_line(ggplot2::aes(y = y, colour = "data"), linewidth = 0.8) +
#     ggplot2::scale_color_manual(
#       name = "", breaks = c("data","median"),
#       values = c(data = "#111111", median = fill_col),
#       labels = c("observed y","predictive median")
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# ## ================================================================
# ## 3) Load data (robust path resolution + header sanitization)
# ## ================================================================
# csv_candidates <- c(
#   "C:/Users/anton/Downloads/data_USGS_ppt_soil.csv",
#   "/mnt/c/Users/anton/Downloads/data_USGS_ppt_soil.csv",
#   path.expand("~/Downloads/data_USGS_ppt_soil.csv"),
#   file.path(getwd(), "data_USGS_ppt_soil.csv")
# )

# csv_path <- NULL
# for (p in csv_candidates) {
#   if (file.exists(p)) { csv_path <- p; break }
# }
# if (is.null(csv_path)) {
#   cat("Working directory:", getwd(), "\n")
#   cat("Checked the following paths and could not find the file:\n",
#       paste0(" - ", csv_candidates, collapse = "\n"), "\n")
#   stop("Could not locate 'data_USGS_ppt_soil.csv'. Please copy it into your project folder ",
#        "or adjust one of the candidate paths above.")
# } else {
#   message("Using data file: ", csv_path)
# }

# # Read and sanitize header so downstream code sees usgs/ppt/soil
# dat <- read.csv(csv_path, check.names = FALSE, stringsAsFactors = FALSE)
# nm <- names(dat)
# nm <- gsub("\ufeff", "", nm, fixed = TRUE)
# nm <- trimws(nm)
# nm <- tolower(nm)
# names(dat) <- nm

# syn_map <- c("precip" = "ppt", "prcp" = "ppt", "rain" = "ppt",
#              "soil_moisture" = "soil", "soilmoist" = "soil", "sm" = "soil")
# for (old in names(syn_map)) {
#   if (old %in% names(dat)) names(dat)[names(dat) == old] <- syn_map[[old]]
# }

# need_cols <- c("usgs", "ppt", "soil")
# if (!all(need_cols %in% names(dat))) {
#   stop("Expected columns: ", paste(need_cols, collapse = ", "),
#        "\nBut found: ", paste(names(dat), collapse = ", "),
#        "\nFix the CSV headers or extend the synonym map above.")
# }
# for (c in need_cols) dat[[c]] <- as.numeric(dat[[c]])
# dat <- tibble::as_tibble(dat)
# message("Columns after sanitization: ", paste(names(dat), collapse = ", "))
# print(utils::head(dat[, need_cols], 5))

# # Make explicit local vectors
# y    <- dat$usgs
# ppt  <- dat$ppt
# soil <- dat$soil

# ## ================================================================
# ## 4) Build covariate lag blocks (full length; align later)
# ## ================================================================
# X_ppt  <- lag_mat(ppt,  lags_ppt)
# X_soil <- lag_mat(soil, lags_soil)
# X_cov_full <- cbind(X_ppt, X_soil)
# maxlag_cov <- max(lags_ppt, lags_soil)

# ## ================================================================
# ## 5) Precompute reservoir once (cheap fit), then per-p readout
# ## ================================================================
# fit0_base <- timer("reservoir states (cheap)", do.call(
#   exdqlm:::qdesn_fit_vb,
#   c(list(
#       y = y, p0 = 0.5,
#       vb_args = list(max_iter = 5, tol = 1e-1, n_samp_xi = 10, verbose = FALSE)
#     ),
#     desn_args)
# ))

# keep_idx <- fit0_base$meta$keep_idx
# y_fit0   <- fit0_base$y_fit
# X_res0   <- fit0_base$X

# # Trim early rows so exogenous lags exist
# trim_n <- sum(keep_idx <= maxlag_cov)
# if (trim_n > 0) {
#   keep_idx <- keep_idx[-seq_len(trim_n)]
#   y_fit0   <- y_fit0[-seq_len(trim_n)]
#   X_res0   <- X_res0[-seq_len(trim_n), , drop = FALSE]
# }
# X_cov   <- X_cov_full[keep_idx, , drop = FALSE]
# X_aug0  <- cbind(X_res0, X_cov)
# T_aligned <- nrow(X_aug0)

# message("X_res0: ", paste(dim(X_res0), collapse="x"),
#         " | X_cov: ", paste(dim(X_cov), collapse="x"),
#         " | X_aug0: ", paste(dim(X_aug0), collapse="x"))

# fit_readout_for_p0 <- function(p0) {
#   fit_readout <- exdqlm:::exal_static_LDVB(
#     y = y_fit0, X = X_aug0, p0 = p0,
#     max_iter = vb_args$max_iter, tol = vb_args$tol,
#     b0 = rep(0, ncol(X_aug0)),
#     V0 = diag(1e4, ncol(X_aug0)),
#     a_sigma = 1e-3, b_sigma = 1e-3,
#     n_samp_xi = vb_args$n_samp_xi,
#     verbose   = isTRUE(vb_args$verbose)
#   )
#   mu_df <- mu_band(X_aug0, fit_readout$qbeta, level = 0.95)
#   df_mu <- tibble::tibble(
#     t_aligned = seq_len(T_aligned),
#     y  = y_fit0,
#     mu = mu_df$mu, lo = mu_df$lo, hi = mu_df$hi, p0 = p0
#   )
#   fit_exog <- list(
#     fit = fit_readout,
#     X   = X_aug0,
#     y_fit = y_fit0,
#     mu_hat = mu_df$mu,
#     reservoir = fit0_base$reservoir,
#     states    = fit0_base$states,
#     meta = list(
#       keep_idx = keep_idx,
#       drop     = max(desn_args$m, desn_args$washout, maxlag_cov),
#       T        = length(y),
#       p0       = p0,
#       D        = desn_args$D,
#       n        = desn_args$n,
#       n_tilde  = desn_args$n_tilde,
#       m        = desn_args$m,
#       alpha    = desn_args$alpha,
#       rho      = desn_args$rho,
#       add_bias = desn_args$add_bias
#     )
#   )
#   class(fit_exog) <- "qdesn_fit"
#   list(fit = fit_exog, df_mu = df_mu)
# }

# # Sequential (simple & deterministic)
# fits <- timer("VB readouts (all p)", lapply(p_vec, fit_readout_for_p0))
# names(fits) <- paste0("p=", p_vec)

# ## ================================================================
# ## 6) Plot per-quantile (μ-band vs observed y)
# ## ================================================================
# plots_mu <- lapply(seq_along(p_vec), function(k) {
#   p0 <- p_vec[k]
#   plot_mu_vs_y(fits[[k]]$df_mu, p0, window = last_window)
# })
# for (g in plots_mu) print(g)

# ## ================================================================
# ## 7) Posterior predictive draws per model (for synthesis)
# ## ================================================================
# pp_draws <- timer("posterior draws (all p)", lapply(fits, function(x) {
#   exdqlm:::posterior_predict.qdesn_fit(x$fit, nd = nd_draws, chunk = chunk_sz)$yrep
# }))

# # All fits share the same alignment
# keep_idx  <- fits[[1]]$fit$meta$keep_idx
# y_aligned <- y[keep_idx]
# T_aligned <- length(keep_idx)

# ## ================================================================
# ## 8) Synthesize across quantile models
# ## ================================================================
# synth <- timer("synthesis", exdqlm:::exdqlm_synthesize_from_draws(
#   draws_list = pp_draws,
#   p          = p_vec,
#   enforce_isotonic = synth_isotonic,
#   rearrange        = synth_rearrange,
#   grid_M           = synth_grid_M,
#   n_samp           = synth_nsamp,
#   seed             = synth_seed,
#   T_expected       = T_aligned
# ))

# ## ================================================================
# ## 9) Observed y with synthesized 95% predictive band
# ## ================================================================
# timer("plot predictive band", {
#   g_band <- plot_obs_with_95_band(
#     synth_draws = synth$draws,
#     y_vec       = y_aligned,
#     window      = last_window,
#     fill_col    = "#3B82F6",
#     show_median = TRUE
#   )
#   print(g_band)
# })


In [ ]:
# ## ================================================================
# ## 10) Fit summary per quantile (gamma, sigma, coverage, band width)
# ## ================================================================
# # Robust extractors for gamma_hat and sigma_hat from a VB fit
# .vb_point_gamma_sigma <- function(fk) {
#   # fk is fits[[k]]
#   qsg <- fk$fit$fit$qsiggam
#   bnd <- fk$fit$fit$misc$bounds
#   # Fallbacks if names differ
#   eta_hat <- if (!is.null(qsg$eta_hat)) qsg$eta_hat else qsg$mu[1]
#   ell_hat <- if (!is.null(qsg$ell_hat)) qsg$ell_hat else qsg$mu[2]
#   L <- if (!is.null(bnd["L"])) as.numeric(bnd["L"]) else 0
#   U <- if (!is.null(bnd["U"])) as.numeric(bnd["U"]) else 1
#   gamma_hat <- L + (U - L) * plogis(eta_hat)
#   sigma_hat <- exp(ell_hat)
#   c(gamma_hat = gamma_hat, sigma_hat = sigma_hat)
# }

# summ_tbl <- dplyr::bind_rows(lapply(seq_along(p_vec), function(k) {
#   fk <- fits[[k]]
#   pars <- .vb_point_gamma_sigma(fk)
#   # Simple diagnostic: fraction of y within the 95% μ-band (not predictive coverage)
#   df_mu <- fk$df_mu
#   in_band <- mean(df_mu$y >= df_mu$lo & df_mu$y <= df_mu$hi, na.rm = TRUE)
#   tibble::tibble(
#     p              = p_vec[k],
#     T_used         = nrow(df_mu),
#     band_w95_mean  = mean(df_mu$hi - df_mu$lo, na.rm = TRUE),
#     mu95_inclusion = in_band,
#     gamma_hat      = pars[["gamma_hat"]],
#     sigma_hat      = pars[["sigma_hat"]]
#   )
# }))

# # nice printing
# summ_tbl <- summ_tbl |>
#   dplyr::mutate(p = scales::percent(p, accuracy = 1))

# print(summ_tbl)

# ## ================================================================
# ## 11) Overlay: μ_t (95% bands) for all p on one plot + observed y
# ##                 -- VISIBILITY-ENHANCED VERSION (with p-label fix)
# ## ================================================================
# # Rebuild the long DF (ensure equal length across fits)
# T_common <- min(sapply(fits, function(x) nrow(x$df_mu)))
# mu_long <- dplyr::bind_rows(lapply(seq_along(p_vec), function(k) {
#   d <- fits[[k]]$df_mu[seq_len(T_common), , drop = FALSE]
#   d$p_chr <- as.character(p_vec[k])
#   d
# }))

# # Colors (keys use two decimals!)
# col_line <- c("0.05" = "#8B0000",   # dark red
#               "0.50" = "#006400",   # dark green
#               "0.95" = "#0F2E6E")   # dark blue
# lab_p <- function(x_chr) scales::percent(as.numeric(x_chr), accuracy = 1)

# slice_window_df <- function(df, N) {
#   i2 <- max(df$t_aligned)
#   i1 <- max(1L, i2 - as.integer(N) + 1L)
#   dplyr::filter(df, dplyr::between(t_aligned, i1, i2))
# }

# plot_overlay_all_mu <- function(df_long, window = 50L) {
#   # Build bg y from any quantile (identical across p by construction)
#   y_df <- df_long |>
#     dplyr::filter(p_chr == unique(df_long$p_chr)[1]) |>
#     dplyr::select(t_aligned, y) |>
#     dplyr::rename(y_bg = y) |>
#     dplyr::distinct(t_aligned, .keep_all = TRUE)

#   dfj <- dplyr::left_join(df_long, y_df, by = "t_aligned") |>
#     dplyr::arrange(p_chr, t_aligned)

#   # Normalize p labels so they match the palette keys (e.g., "0.50")
#   dfj$p_chr <- factor(sprintf("%.2f", as.numeric(dfj$p_chr)),
#                       levels = names(col_line))

#   # Diagnostic: band width in the window (optional)
#   d_diag <- slice_window_df(dfj, window)
#   bw_med <- d_diag |>
#     dplyr::mutate(bw = hi - lo) |>
#     dplyr::summarise(median_bw = median(bw, na.rm = TRUE)) |>
#     dplyr::pull(median_bw)
#   message(sprintf("Median band width in window: %.4f", bw_med))

#   d <- d_diag

#   ggplot2::ggplot(d, ggplot2::aes(x = t_aligned)) +
#     ggplot2::theme_minimal(base_size = 13) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "time (aligned)", y = "USGS (response)",
#       title = "All quantile fits: μ̂ with 95% credible bands + observed data",
#       subtitle = sprintf("Quantiles: %s; last %d points",
#                          paste(lab_p(levels(d$p_chr)), collapse = ", "),
#                          length(unique(d$t_aligned)))
#     ) +
#     # ribbons first
#     ggplot2::geom_ribbon(
#       ggplot2::aes(ymin = lo, ymax = hi, fill = p_chr, group = p_chr),
#       alpha = 0.28, colour = NA
#     ) +
#     # band boundaries (dotted)
#     ggplot2::geom_line(
#       ggplot2::aes(y = lo, colour = p_chr, group = p_chr),
#       linewidth = 0.35, linetype = "dotted", alpha = 0.6
#     ) +
#     ggplot2::geom_line(
#       ggplot2::aes(y = hi, colour = p_chr, group = p_chr),
#       linewidth = 0.35, linetype = "dotted", alpha = 0.6
#     ) +
#     # observed y in the background
#     ggplot2::geom_line(
#       ggplot2::aes(x = t_aligned, y = y_bg),
#       linewidth = 0.7, colour = "#222222", alpha = 0.65, inherit.aes = FALSE
#     ) +
#     ggplot2::geom_point(
#       ggplot2::aes(x = t_aligned, y = y_bg),
#       size = 0.6, colour = "#444444", alpha = 0.35, inherit.aes = FALSE
#     ) +
#     # μ̂ lines and points
#     ggplot2::geom_line(ggplot2::aes(y = mu, colour = p_chr, group = p_chr), linewidth = 0.95) +
#     ggplot2::geom_point(ggplot2::aes(y = mu, colour = p_chr, group = p_chr), size = 0.8, alpha = 0.9) +
#     ggplot2::scale_color_manual(name = "quantile p",
#                                 values = col_line,
#                                 labels = lab_p,
#                                 drop = FALSE) +
#     ggplot2::scale_fill_manual(name = "quantile p",
#                                values = scales::alpha(col_line, 0.28),
#                                labels = lab_p,
#                                drop = FALSE) +
#     ggplot2::guides(
#       colour = ggplot2::guide_legend(order = 1, override.aes = list(size = 3)),
#       fill   = ggplot2::guide_legend(order = 2, override.aes = list(alpha = 0.28))
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# # Build and show overlay plot
# g_all <- plot_overlay_all_mu(mu_long, window = 100L)
# print(g_all)


In [ ]:
# ## ================================================================
# ## 12) Quantile calibration of point predictors μ_t
# ##     Pr(y_t ≤ μ_t) ≈ p0  + rolling coverage diagnostics
# ## ================================================================

# # --- helpers ------------------------------------------------------
# # Wilson binomial interval (better than normal approx when p near 0/1)
# wilson_ci <- function(k, n, conf = 0.95) {
#   if (n <= 0) return(c(NA_real_, NA_real_))
#   z <- qnorm(0.5 + conf/2)
#   p <- k / n
#   den <- 1 + z^2 / n
#   cen <- (p + z^2/(2*n)) / den
#   rad <- z * sqrt(p*(1-p)/n + z^2/(4*n^2)) / den
#   c(max(0, cen - rad), min(1, cen + rad))
# }

# # Pinball (quantile) loss
# pinball_loss <- function(y, qhat, p) {
#   e <- y - qhat
#   (p - (e < 0)) * e
# }

# # Simple trailing rolling mean (length W), padding with NA for first W-1
# roll_mean <- function(x, W) {
#   if (W <= 1) return(x)
#   as.numeric(stats::filter(x, rep(1 / W, W), sides = 1))
# }

# # ensure color map exists (matches Part 11)
# if (!exists("col_line", inherits = FALSE)) {
#   col_line <- c("0.05" = "#8B0000", "0.50" = "#006400", "0.95" = "#0F2E6E")
# }

# # --- GLOBAL coverage & loss per quantile --------------------------
# cov_tbl <- dplyr::bind_rows(lapply(seq_along(fits), function(i) {
#   df <- fits[[i]]$df_mu
#   # p0 stored in df_mu by construction
#   p0 <- as.numeric(df$p0[1])
#   # keep only rows with finite y & mu
#   ok <- is.finite(df$y) & is.finite(df$mu)
#   ind <- as.integer(df$y[ok] <= df$mu[ok])
#   k <- sum(ind, na.rm = TRUE); n <- sum(!is.na(ind))
#   ci <- wilson_ci(k, n, conf = 0.95)
#   tibble::tibble(
#     p0         = p0,
#     N          = n,
#     coverage   = if (n > 0) k / n else NA_real_,
#     cov_lo95   = ci[1],
#     cov_hi95   = ci[2],
#     pinball    = mean(pinball_loss(df$y[ok], df$mu[ok], p0), na.rm = TRUE)
#   )
# })) |>
#   dplyr::arrange(p0)

# print(cov_tbl)

# # --- Rolling coverage curves --------------------------------------
# cov_window <- 365L   # trailing window size; change to taste
# cov_long <- dplyr::bind_rows(lapply(seq_along(fits), function(i) {
#   df <- fits[[i]]$df_mu
#   p0 <- as.numeric(df$p0[1])
#   p_chr <- sprintf("%.2f", p0)
#   ok <- is.finite(df$y) & is.finite(df$mu)
#   ind <- rep(NA_integer_, nrow(df))
#   ind[ok] <- as.integer(df$y[ok] <= df$mu[ok])
#   rcov <- roll_mean(ind, cov_window)
#   tibble::tibble(
#     t_aligned = df$t_aligned,
#     p_chr     = p_chr,
#     p0        = p0,
#     rcov      = rcov
#   )
# }))

# plot_rolling_cov <- function(df_long, window = 10000L, show_last = 500L) {
#   d <- df_long |>
#     dplyr::group_by(p_chr) |>
#     dplyr::mutate(t_max = max(t_aligned, na.rm = TRUE)) |>
#     dplyr::ungroup() |>
#     dplyr::filter(t_aligned > (t_max - show_last)) |>
#     dplyr::arrange(p_chr, t_aligned)

#   # per-quantile fixed reference bands (normal approximation for visualization)
#   bands <- d |>
#     dplyr::distinct(p_chr, p0) |>
#     dplyr::rowwise() |>
#     dplyr::mutate(
#       se = sqrt(p0 * (1 - p0) / window),
#       lo = p0 - qnorm(0.975) * se,
#       hi = p0 + qnorm(0.975) * se
#     ) |>
#     dplyr::ungroup()

#   # x-span for the shaded band (entire plotted window)
#   t_min <- min(d$t_aligned, na.rm = TRUE)
#   t_max <- max(d$t_aligned, na.rm = TRUE)
#   bands <- bands |>
#     dplyr::mutate(xmin = t_min, xmax = t_max)

#   ggplot2::ggplot(d, ggplot2::aes(x = t_aligned, y = rcov, colour = p_chr)) +
#     ggplot2::theme_minimal(base_size = 13) +
#     ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
#                    legend.position = "right") +
#     ggplot2::labs(
#       x = "time (aligned)",
#       y = sprintf("rolling Pr(y ≤ μ)  (W = %d)", window),
#       title = "Rolling empirical coverage of μ_t",
#       subtitle = sprintf("Last %d points; shaded bands are ±1.96√(p(1-p)/W) reference", show_last)
#     ) +
#     # target p lines
#     ggplot2::geom_hline(data = bands,
#                         ggplot2::aes(yintercept = p0, colour = p_chr),
#                         linetype = "dashed", linewidth = 0.7, show.legend = FALSE) +
#     # reference CI band (rectangles across the plotting window)
#     ggplot2::geom_rect(data = bands,
#                        ggplot2::aes(xmin = xmin, xmax = xmax, ymin = lo, ymax = hi, fill = p_chr),
#                        inherit.aes = FALSE, alpha = 0.12) +
#     # rolling coverage curve
#     ggplot2::geom_line(linewidth = 0.9, na.rm = TRUE) +
#     ggplot2::scale_color_manual(
#       name = "quantile p",
#       values = col_line,
#       labels = function(x) scales::percent(as.numeric(x))
#     ) +
#     ggplot2::scale_fill_manual(
#       name = "quantile p",
#       values = sapply(col_line, scales::alpha, alpha = 0.12),
#       labels = function(x) scales::percent(as.numeric(x))
#     ) +
#     ggplot2::coord_cartesian(expand = FALSE)
# }

# g_cov <- plot_rolling_cov(cov_long, window = cov_window, show_last = 300L)
# print(g_cov)
